# Preparing the datasets

Source: [LinkedIn Learning - Build A SQL AI Agent for Production](https://www.linkedin.com/learning/build-with-ai-sql-ai-agents-in-production)

Throughout the course, we will use the San Francisco International Airport report on monthly passenger traffic statistics by airline dataset. In this lesson we will:
- Load and review the dataset
- Prepare the dataset to work with AI agents
- Review the DuckDB approach
- Review the PostgreSQL approach 

This dataset is available at [here](https://data.sfgov.org/Transportation/Air-Traffic-Passenger-Statistics/rkru-6vcg/about_data)

## Loading the Air Passenger Traffic Dataset

Let's start by import the required libraries:

In [1]:
import pandas as pd

In [2]:
file_path = "../data/Air_Traffic_Passenger_Statistics_20260201.csv"
df = pd.read_csv(file_path)
df.head()

,Activity Period,Activity Period Start Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count,data_as_of,data_loaded_at
0,199907,7/1/99,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432,1/20/26 14:01,1/22/26 15:02
1,199907,7/1/99,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353,1/20/26 14:01,1/22/26 15:02
2,199907,7/1/99,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518,1/20/26 14:01,1/22/26 15:02
3,199907,7/1/99,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Deplaned,Other,Terminal 2,D,1324,1/20/26 14:01,1/22/26 15:02
4,199907,7/1/99,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Enplaned,Other,Terminal 2,D,1198,1/20/26 14:01,1/22/26 15:02


Let's prep the dataset to work wi with AI agents:
- Validate the column names
- Rename the column names
- Remove irrelevant columns
- Reformat the columns


In [3]:
df["Date"] = pd.to_datetime(df["Activity Period Start Date"], format="%m/%d/%y")

df["Year"] = df["Activity Period"].astype(str).str[:4].astype(int)

columns = [
    "Year",
    "Date",
    "Operating Airline",
    "Operating Airline IATA Code",
    "Published Airline",
    "Published Airline IATA Code",
    "GEO Summary",
    "GEO Region",
    "Activity Type Code",
    "Price Category Code",
    "Terminal",
    "Boarding Area",
    "Passenger Count"
]

air_traffic = df[columns].copy()
air_traffic.dtypes

Year                                    int64
Date                           datetime64[ns]
Operating Airline                      object
Operating Airline IATA Code            object
Published Airline                      object
Published Airline IATA Code            object
GEO Summary                            object
GEO Region                             object
Activity Type Code                     object
Price Category Code                    object
Terminal                               object
Boarding Area                          object
Passenger Count                         int64
dtype: object

In [4]:
air_traffic

,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,1999,1999-07-01,Aeroflot Russian International Airlines,NaN,Aeroflot Russian International Airlines,NaN,International,Europe,Enplaned,Other,Terminal 2,D,1198
...,...,...,...,...,...,...,...,...,...,...,...,...,...
39227,2025,2025-11-01,Virgin Atlantic,VS,Virgin Atlantic,VS,International,Europe,Enplaned,Other,International,A,7109
39228,2025,2025-11-01,WestJet,WS,WestJet,WS,International,Canada,Deplaned,Other,International,A,2445
39229,2025,2025-11-01,WestJet,WS,WestJet,WS,International,Canada,Enplaned,Other,International,A,2487
39230,2025,2025-11-01,ZIPAIR Tokyo Inc,ZG,ZIPAIR Tokyo Inc,ZG,International,Asia,Deplaned,Other,International,A,7279


In [5]:
air_traffic.to_csv("../data/air_traffic_gold.csv")

In [6]:
tbl_name = "air_traffic"

## DuckDB Workflow

In this section, we will review how to set up in-memory DuckDB database using the `ibis` library. Let's start by loading the `ibis` library:

In [7]:
import ibis

In [8]:
con_db = ibis.duckdb.connect()
con_db.create_table(tbl_name, air_traffic, overwrite=True)


DatabaseTable: memory.main.air_traffic
  Year                        int64
  Date                        timestamp(6)
  Operating Airline           string
  Operating Airline IATA Code string
  Published Airline           string
  Published Airline IATA Code string
  GEO Summary                 string
  GEO Region                  string
  Activity Type Code          string
  Price Category Code         string
  Terminal                    string
  Boarding Area               string
  Passenger Count             int64

In [9]:
con_db.sql("SELECT * FROM air_traffic LIMIT 10").execute()


,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Enplaned,Other,Terminal 2,D,1198
5,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Deplaned,Other,Terminal 1,B,24124
6,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Enplaned,Other,Terminal 1,B,23613
7,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Deplaned,Other,Terminal 2,D,4983
8,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Enplaned,Other,Terminal 2,D,4604
9,1999,1999-07-01,Air Europe,PE,Air Europe,PE,International,Europe,Deplaned,Other,Terminal 2,D,205


## Postgres Workflow

In [10]:
con_postgres = ibis.postgres.connect(
    user="postgres",
    password="password",
    host="postgres",
    port=5432,
    database="my_db",
)

In [11]:
schema = ibis.memtable(air_traffic).schema()
print(schema)

ibis.Schema {
  Year                         int64
  Date                         timestamp
  Operating Airline            string
  Operating Airline IATA Code  string
  Published Airline            string
  Published Airline IATA Code  string
  GEO Summary                  string
  GEO Region                   string
  Activity Type Code           string
  Price Category Code          string
  Terminal                     string
  Boarding Area                string
  Passenger Count              int64
}


In [12]:
con_postgres.create_table("air_traffic", air_traffic, schema=schema, overwrite=True)

DatabaseTable: air_traffic
  Year                        int64
  Date                        timestamp
  Operating Airline           string
  Operating Airline IATA Code string
  Published Airline           string
  Published Airline IATA Code string
  GEO Summary                 string
  GEO Region                  string
  Activity Type Code          string
  Price Category Code         string
  Terminal                    string
  Boarding Area               string
  Passenger Count             int64

In [13]:
con_postgres.sql("SELECT * FROM air_traffic LIMIT 10").execute()


,Year,Date,Operating Airline,Operating Airline IATA Code,Published Airline,Published Airline IATA Code,GEO Summary,GEO Region,Activity Type Code,Price Category Code,Terminal,Boarding Area,Passenger Count
0,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Deplaned,Low Fare,Terminal 1,B,31432
1,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Enplaned,Low Fare,Terminal 1,B,31353
2,1999,1999-07-01,ATA Airlines,TZ,ATA Airlines,TZ,Domestic,US,Thru / Transit,Low Fare,Terminal 1,B,2518
3,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Deplaned,Other,Terminal 2,D,1324
4,1999,1999-07-01,Aeroflot Russian International Airlines,None,Aeroflot Russian International Airlines,None,International,Europe,Enplaned,Other,Terminal 2,D,1198
5,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Deplaned,Other,Terminal 1,B,24124
6,1999,1999-07-01,Air Canada,AC,Air Canada,AC,International,Canada,Enplaned,Other,Terminal 1,B,23613
7,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Deplaned,Other,Terminal 2,D,4983
8,1999,1999-07-01,Air China,CA,Air China,CA,International,Asia,Enplaned,Other,Terminal 2,D,4604
9,1999,1999-07-01,Air Europe,PE,Air Europe,PE,International,Europe,Deplaned,Other,Terminal 2,D,205


## Database Logging with Ibis - PostgreSQL and DuckDB
In this section, we will demonstrate how to set up database logging for the SQL AI Agent using the get_ibis_connection() utility. We will:

- Create a helper function to convert JSON log files to CSV for DuckDB
- Initialize PostgreSQL log table using get_ibis_connection()
- Register log table with DuckDB using CSV conversion
- Generate sample logs and query them from both databases
- This approach provides a unified workflow for both databases using the same connection utility.
  

In [15]:
import sys
import os

# Add project root to path
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)


In [16]:
import pandas as pd
import json
import logging
from datetime import datetime
import ibis

# SQL AI Agent imports
from sql_ai_agent.data import get_ibis_connection
from sql_ai_agent.log_database import (
    init_postgres_log_schema,
    init_duckdb_log_schema,
    DatabaseLogHandler,
    verify_log_table,
)
from sql_ai_agent.logger import SQLAgentLogger

print("✅ All imports successful!")


✅ All imports successful!


### Helper Function: Convert JSON Logs to CSV

Since DuckDB works well with CSV files through get_ibis_connection(), we need a function to convert JSON log files to CSV format:

In [17]:
def json_logs_to_csv(json_file_path: str, csv_file_path: str) -> pd.DataFrame:
    """
    Convert JSON log file to CSV format for DuckDB ingestion.

    Args:
        json_file_path: Path to JSON log file (one JSON object per line)
        csv_file_path: Path to output CSV file

    Returns:
        DataFrame with processed log data
    """
    # Read JSON log file (newline-delimited JSON)
    logs = []
    with open(json_file_path, "r") as f:
        for line in f:
            try:
                log_entry = json.loads(line.strip())
                logs.append(log_entry)
            except json.JSONDecodeError:
                continue  # Skip invalid lines

    if not logs:
        raise ValueError("No valid JSON log entries found")

    # Convert to DataFrame
    df = pd.DataFrame(logs)

    # Ensure all required columns exist
    required_columns = [
        "timestamp",
        "level",
        "logger",
        "message",
        "session_id",
        "operation_type",
    ]

    for col in required_columns:
        if col not in df.columns:
            df[col] = None

    # Handle extra fields - convert dict/list columns to JSON strings
    extra_fields_data = []
    for idx, row in df.iterrows():
        extra_dict = {}
        for col in df.columns:
            if col not in required_columns + ["exception", "extra_fields"]:
                if pd.notna(row[col]):
                    extra_dict[col] = row[col]

        # Also include existing extra_fields if present
        if "extra_fields" in row and isinstance(row["extra_fields"], dict):
            extra_dict.update(row["extra_fields"])

        extra_fields_data.append(
            json.dumps(extra_dict) if extra_dict else None
        )

    # Create final DataFrame with schema matching database
    result_df = pd.DataFrame(
        {
            "timestamp": pd.to_datetime(df["timestamp"]),
            "level": df["level"],
            "logger": df["logger"],
            "message": df["message"],
            "session_id": df["session_id"],
            "operation_type": df["operation_type"],
            "extra_fields": extra_fields_data,
            "exception": df.get("exception", None),
        }
    )

    # Save to CSV
    result_df.to_csv(csv_file_path, index=False)
    print(f"✅ Converted {len(result_df)} log entries to CSV: {csv_file_path}")

    return result_df


print("✅ Helper function defined")


✅ Helper function defined


### PostgreSQL Workflow
In this section, we will:

- Connect to PostgreSQL using get_ibis_connection()
- Initialize the log schema
- Set up database logging
- Generate sample logs
- Query the logs

#### Step 1: Configure PostgreSQL Connection

In [18]:
# PostgreSQL configuration
postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "postgres",
    "port": 5432,
    "database": "my_db",
}

# Table name for logs
log_table_name = "sql_agent_logs"

print("✅ PostgreSQL configuration set")


✅ PostgreSQL configuration set


#### Step 2: Connect to PostgreSQL

**Note:** For this workflow, we'll use direct ibis.postgres.connect() since get_ibis_connection() is designed for loading data tables, not for administrative tasks like creating log schemas.

In [19]:
# Initialize log table with optimized schema
init_postgres_log_schema(
    con=con_postgres, table_name=log_table_name, schema="public"
)

print(f"\n✅ PostgreSQL log table initialized!")
print(f"   Table: public.{log_table_name}")
print(f"   Indexes: 6 created for query performance")


✅ PostgreSQL log table '"public"."sql_agent_logs"' initialized successfully
   - Table created with optimized schema
   - 6 indexes created for query performance
   - Ready to receive log entries

✅ PostgreSQL log table initialized!
   Table: public.sql_agent_logs
   Indexes: 6 created for query performance


#### Step 4: Verify Table Creation

In [20]:
# Verify table exists
stats = verify_log_table(con_postgres, log_table_name, "postgres")

print("📊 Table Statistics:")
print(f"   Table exists: {stats['exists']}")
print(f"   Total logs: {stats['row_count']}")


📊 Table Statistics:
   Table exists: True
   Total logs: 0


#### Step 5: Set Up Database Logging Handler

In [21]:
# Create logger for PostgreSQL
pg_logger = logging.getLogger("sql_ai_agent_postgres")
pg_logger.setLevel(logging.INFO)
pg_logger.handlers.clear()

# Add database handler
pg_db_handler = DatabaseLogHandler(
    con=con_postgres,
    table_name=log_table_name,
    db_type="postgres",
    schema="public",
    level=logging.INFO,
)
pg_logger.addHandler(pg_db_handler)

# Add console handler for visibility
console_handler = logging.StreamHandler()
console_handler.setFormatter(
    logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
)
pg_logger.addHandler(console_handler)

print("✅ PostgreSQL database logging configured!")


✅ PostgreSQL database logging configured!
